In [1]:
# ==========================================
# Cell 1: Environment Setup & Installs
# ==========================================

# Install necessary dependencies. 
# (Note: Remove the '#' on the line below if you are running this for the first time)
# !pip install -q "timesfm[torch,xreg]" plotly fredapi python-dotenv pandas numpy google-generativeai

import os
import json
import warnings
import pandas as pd
import numpy as np

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# APIs & Secrets
from fredapi import Fred
from dotenv import load_dotenv

# Machine Learning & AI
import timesfm

# Suppress warnings for a clean notebook output
warnings.filterwarnings('ignore')

print("Cell 1 Execution Complete: Libraries imported successfully. Environment is ready.")

Cell 1 Execution Complete: Libraries imported successfully. Environment is ready.


In [ ]:
# ==========================================
# Cell 2: Configuration & Secret Management
# ==========================================

# Load environment variables from a .env file
# Ensure your .env file contains: FRED_API_KEY=your_actual_key_here
load_dotenv()

# Initialize FRED API Client
fred_api_key = os.getenv("FRED_API_KEY")
if not fred_api_key:
    raise ValueError("FRED_API_KEY not found. Please ensure it is set in your .env file.")
fred = Fred(api_key=fred_api_key)

# Global Pipeline Parameters
TARGET_SERIES = 'DREQRG'          # PCE Index for tech/media equipment
COVARIATE_SERIES = 'POILBREUSDM'  # Brent Crude Oil Prices
HORIZON_LEN = 12                  # 12-month forecast horizon (May 2026 - May 2027)
CONTEXT_LEN = 120                 # 10-year lookback for TimesFM context

print("Cell 2 Execution Complete: Configuration loaded and FRED API authenticated.")

Cell 2 Execution Complete: Configuration loaded and FRED API authenticated.


In [3]:
# ==========================================
# Cell 3: Target Data Extraction (BEA Data)
# ==========================================

print("Loading and parsing BEA target data...")

file_path = "Section2All_xls - U20304-M.csv"

# 1. Load the raw data, skipping the first 7 rows of text metadata
# The 8th row (index 7) contains the actual column headers (Line, Unnamed, Unnamed, 1959M01, ...)
try:
    df_raw = pd.read_csv(file_path, skiprows=7)
except FileNotFoundError:
    raise FileNotFoundError(f"Could not find '{file_path}'. Please ensure it is in the same directory as this notebook.")

# 2. Isolate the target row (Series Code: DREQRG)
# In this specific BEA format, the series code lives in the 3rd column (index 2)
mask = df_raw.iloc[:, 2] == TARGET_SERIES

if not mask.any():
    # Fallback: search the entire dataframe just in case the format shifts slightly
    mask = df_raw.apply(lambda row: row.astype(str).str.contains(TARGET_SERIES).any(), axis=1)
    if not mask.any():
        raise ValueError(f"Series Code '{TARGET_SERIES}' not found in the dataset.")

target_row = df_raw[mask].iloc[0]

# 3. Extract just the time series data (ignoring the first 3 metadata columns: Line, Desc, Code)
ts_data = target_row.iloc[3:]

# 4. Create a fresh vertical DataFrame
df_pce = pd.DataFrame({
    'Raw_Date': ts_data.index,
    'PCE_Index': ts_data.values
})

# 5. Clean and parse dates
# Replace 'M' with '-' (e.g., '1959M01' becomes '1959-01')
df_pce['Clean_Date'] = df_pce['Raw_Date'].astype(str).str.replace('M', '-', regex=False)

# Convert to datetime objects and strictly anchor them to End-of-Month (EOM)
df_pce['Date'] = pd.to_datetime(df_pce['Clean_Date'], errors='coerce') + pd.offsets.MonthEnd(0)

# Ensure the PCE index values are numeric floats, turning any leftover text into NaN
df_pce['PCE_Index'] = pd.to_numeric(df_pce['PCE_Index'], errors='coerce')

# 6. Final Cleanup
# Drop any invalid dates/values, keep only what we need, and sort chronologically
df_target = df_pce.dropna(subset=['Date', 'PCE_Index'])[['Date', 'PCE_Index']].sort_values('Date').reset_index(drop=True)

print("Target data successfully processed.")
print(f"Rows extracted: {len(df_target)}")
if len(df_target) > 0:
    print(f"Date Range: {df_target['Date'].min().date()} to {df_target['Date'].max().date()}")

# Display the last 5 rows to visually verify it reaches April/May 2026
display(df_target.tail())

Loading and parsing BEA target data...
Target data successfully processed.
Rows extracted: 806
Date Range: 1959-01-31 to 2026-02-28


,Date,PCE_Index
801,2025-10-31,84.075
802,2025-11-30,83.886
803,2025-12-31,85.397
804,2026-01-31,86.730
805,2026-02-28,88.681


In [4]:
# ==========================================
# Cell 4: Covariate Data Acquisition (FRED)
# ==========================================

print(f"Fetching covariate data ({COVARIATE_SERIES}) from FRED API...")

try:
    # 1. Attempt API Pull
    brent_raw = fred.get_series(COVARIATE_SERIES)
    
    # 2. Convert to DataFrame
    df_brent = brent_raw.reset_index()
    df_brent.columns = ['Parsed_Date', 'Brent_Crude_Price']
    
    # 3. Anchor to End-of-Month (EOM) to match BEA data
    df_brent['Date'] = df_brent['Parsed_Date'] + pd.offsets.MonthEnd(0)
    
    # 4. Final Cleanup
    df_covariate = df_brent[['Date', 'Brent_Crude_Price']].dropna().sort_values('Date').reset_index(drop=True)
    print("FRED API pull successful.")
    
except Exception as e:
    # Fallback Mechanism
    print(f"WARNING: API Pull failed ({e}). Attempting to load local cache...")
    fallback_path = "brent_crude_backup.csv"
    
    try:
        df_brent = pd.read_csv(fallback_path)
        df_brent['Parsed_Date'] = pd.to_datetime(df_brent['Date'], errors='coerce')
        df_brent['Date'] = df_brent['Parsed_Date'] + pd.offsets.MonthEnd(0)
        df_covariate = df_brent[['Date', 'Brent_Crude_Price']].dropna().sort_values('Date').reset_index(drop=True)
        print(f"Local cache loaded successfully from {fallback_path}.")
    except FileNotFoundError:
        raise FileNotFoundError(f"API failed and fallback cache '{fallback_path}' could not be found. Pipeline halted.")

# Filter covariate to strictly match the historical dates of the target data
start_date = df_target['Date'].min()
end_date = df_target['Date'].max()

df_covariate = df_covariate[(df_covariate['Date'] >= start_date) & (df_covariate['Date'] <= end_date)].reset_index(drop=True)

print(f"Covariate data successfully processed.")
print(f"Rows extracted: {len(df_covariate)}")
if len(df_covariate) > 0:
    print(f"Date Range: {df_covariate['Date'].min().date()} to {df_covariate['Date'].max().date()}")

# Display the last 5 rows for visual verification
display(df_covariate.tail())

Fetching covariate data (POILBREUSDM) from FRED API...
FRED API pull successful.
Covariate data successfully processed.
Rows extracted: 410
Date Range: 1992-01-31 to 2026-02-28


,Date,Brent_Crude_Price
405,2025-10-31,63.981304
406,2025-11-30,63.693500
407,2025-12-31,61.810909
408,2026-01-31,64.594091
409,2026-02-28,69.409500


In [5]:
# ==========================================
# Cell 5: Data Merging & Imputation
# ==========================================

print("Aligning and merging target and covariate datasets...")

# 1. Merge the datasets
# We use an 'outer' join to keep all dates, then sort chronologically.
# This ensures we don't lose the latest BEA data if FRED is lagging, or vice versa.
df_master = pd.merge(df_target, df_covariate, on='Date', how='outer').sort_values('Date').reset_index(drop=True)

# 2. Handle Publication Lags (Imputation)
# Check if there are any missing values in either column.
missing_brent = df_master['Brent_Crude_Price'].isna().sum()
missing_pce = df_master['PCE_Index'].isna().sum()

if missing_brent > 0 or missing_pce > 0:
    print(f"Publication lag detected: Missing {missing_brent} Brent values and {missing_pce} PCE values.")
    print("Applying forward-fill (limit=1) to align current-month data...")
    
    # Forward fill missing values, but only by exactly 1 month. 
    # If data is missing for 2+ months, the pipeline will legitimately fail later, which is safer than deep hallucination.
    df_master['Brent_Crude_Price'] = df_master['Brent_Crude_Price'].ffill(limit=1)
    df_master['PCE_Index'] = df_master['PCE_Index'].ffill(limit=1)

# 3. Final drop of any rows that STILL have NaNs (e.g., historical mismatches early in the series)
df_master = df_master.dropna().reset_index(drop=True)

print("Phase 1 Complete: Master historical dataset is ready for TimesFM.")
print("-" * 50)
print(f"Total Aligned Rows: {len(df_master)}")
print(f"Final Date Range:   {df_master['Date'].min().date()} to {df_master['Date'].max().date()}")
print("-" * 50)

# Display the last 5 rows to confirm alignment
display(df_master.tail())

Aligning and merging target and covariate datasets...
Publication lag detected: Missing 396 Brent values and 0 PCE values.
Applying forward-fill (limit=1) to align current-month data...
Phase 1 Complete: Master historical dataset is ready for TimesFM.
--------------------------------------------------
Total Aligned Rows: 410
Final Date Range:   1992-01-31 to 2026-02-28
--------------------------------------------------


,Date,PCE_Index,Brent_Crude_Price
405,2025-10-31,84.075,63.981304
406,2025-11-30,83.886,63.693500
407,2025-12-31,85.397,61.810909
408,2026-01-31,86.730,64.594091
409,2026-02-28,88.681,69.409500


In [6]:
# ==========================================
# Cell 6: The Qualitative Engine (Manual Inputs)
# ==========================================

print("Phase 2: Loading Qualitative Energy Scenarios...")

# ---------------------------------------------------------
# ANALYST INPUT REQUIRED: Paste your Qualitative Output Below
# ---------------------------------------------------------
# The arrays must contain exactly 13 floats representing the 
# monthly Brent Crude price (USD/bbl) from May 2026 through May 2027.

scenario_inputs = {
    "base_case": [110.50, 115.00, 112.50, 105.00, 95.00, 88.00, 82.00, 78.00, 76.00, 75.00, 74.00, 75.00, 76.00],
    "bull_case": [110.50, 125.00, 138.00, 145.00, 155.00, 150.00, 142.00, 138.00, 135.00, 130.00, 128.00, 125.00, 122.00],
    "bear_case": [110.50, 95.00, 85.00, 78.00, 72.00, 68.00, 65.00, 63.00, 60.00, 58.00, 56.00, 55.00, 55.00],
    "narrative": "The 1990-1991 Gulf War serves as the closest historical analog for the current May 2026 environment, as the active Iran conflict and Strait of Hormuz disruptions mirror a massive but acute Middle Eastern geopolitical supply panic. Just as in 1990, this sharp shock is projected to spike prices aggressively before resolving rapidly within months as military or diplomatic interventions restore baseline flows."
}

# ---------------------------------------------------------
# Edge Case Mitigation: Bounding Logic & Validation
# ---------------------------------------------------------
MIN_PRICE = 20.0
MAX_PRICE = 250.0

# Clip extreme outliers to prevent Out-of-Distribution (OOD) collapse in TimesFM
brent_scenarios = {
    "Base Case": np.clip(scenario_inputs["base_case"], MIN_PRICE, MAX_PRICE),
    "Bull Case": np.clip(scenario_inputs["bull_case"], MIN_PRICE, MAX_PRICE),
    "Bear Case": np.clip(scenario_inputs["bear_case"], MIN_PRICE, MAX_PRICE)
}

# Validate that the arrays contain exactly 13 months
for case_name, array in brent_scenarios.items():
    if len(array) != 13:
        raise ValueError(f"Data Error: '{case_name}' array has {len(array)} values. Expected exactly 13.")

# Output the narrative for the analyst's terminal
print("\n" + "="*60)
print("MACRO NARRATIVE ASSUMPTION:")
print("="*60)
print(scenario_inputs["narrative"])
print("="*60 + "\n")
print("Phase 2 Complete: Scenarios loaded, validated, and clipped to safe bounds ($20 - $250).")

Phase 2: Loading Qualitative Energy Scenarios...

MACRO NARRATIVE ASSUMPTION:
The 1990-1991 Gulf War serves as the closest historical analog for the current May 2026 environment, as the active Iran conflict and Strait of Hormuz disruptions mirror a massive but acute Middle Eastern geopolitical supply panic. Just as in 1990, this sharp shock is projected to spike prices aggressively before resolving rapidly within months as military or diplomatic interventions restore baseline flows.

Phase 2 Complete: Scenarios loaded, validated, and clipped to safe bounds ($20 - $250).


In [11]:
# ==========================================
# Cell 7: TimesFM Initialization (v2.5) - CORRECTED
# ==========================================
import torch
import timesfm

print("Phase 3: Initializing TimesFM 2.5 Quantitative Engine...")

# Optimize matrix multiplications for PyTorch (Recommended by Google Research)
torch.set_float32_matmul_precision("high")

# 1. Initialize the 200M parameter model directly from Hugging Face
print("Loading TimesFM 2.5 model checkpoints from Hugging Face...")
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")

# 2. Compile the model with our specific forecasting configuration
print("Compiling model configuration for XReg...")
tfm.compile(
    timesfm.ForecastConfig(
        max_context=CONTEXT_LEN,       # 120 months (10 years lookback)
        max_horizon=HORIZON_LEN,       # 12 months forward
        normalize_inputs=True,         # Reversible instance normalization (RevIN)
        infer_is_positive=True,        # The PCE Index is strictly positive
        return_backcast=True           # REQUIRED for exogenous regressors (XReg) in v2.5
    )
)

print("Phase 3 - Part 1 Complete: TimesFM 2.5 successfully loaded into memory and compiled.")

Phase 3: Initializing TimesFM 2.5 Quantitative Engine...
Loading TimesFM 2.5 model checkpoints from Hugging Face...


Compiling model configuration for XReg...
Phase 3 - Part 1 Complete: TimesFM 2.5 successfully loaded into memory and compiled.


In [12]:
# ==========================================
# Cell 8: Dynamic Covariate Mapping & Inference
# ==========================================

print("Phase 3 - Part 2: Executing TimesFM Scenarios...")

# 1. Prepare Target Data
# We extract the last CONTEXT_LEN (120) months as a 1D array
target_data = df_master['PCE_Index'].values
context_data = target_data[-CONTEXT_LEN:]

# 2. Prepare Historical Covariate Data
historical_covariate = df_master['Brent_Crude_Price'].values
covariate_context = historical_covariate[-CONTEXT_LEN:]

# Dictionary to store our forecasted trajectories
forecast_results = {}

# 3. Inference Loop
for scenario_name, future_brent in brent_scenarios.items():
    print(f"Running inference for: {scenario_name}...")
    
    # future_brent contains 13 months (May 2026 to May 2027 inclusive).
    # We drop the first month to strictly forecast the *next* 12 months.
    future_covariate = future_brent[1:] 
    
    # Combine past and future into a single 1D array.
    # The covariate array length MUST equal: len(context_data) + HORIZON_LEN
    full_covariate = np.concatenate([covariate_context, future_covariate])
    
    try:
        # Use the specialized covariate forecasting method.
        # We pass inputs as a list of batches (we have 1 batch), and the covariate as a dictionary.
        forecast, _ = tfm.forecast_with_covariates(
            inputs=[context_data], 
            dynamic_numerical_covariates={"brent_price": [full_covariate]}
        )
        
        # Extract the point forecast (median) for our single batch
        point_forecast = forecast[0]
        
        # 4. Out-of-Distribution (OOD) Fallback Logic
        if np.isnan(point_forecast).any():
            print(f"  -> WARNING: {scenario_name} caused model collapse (NaNs detected).")
            print("  -> Triggering standard Univariate Fallback...")
            
            # If the covariate mathematically breaks the model, fallback to a standard forecast
            fallback_forecast, _ = tfm.forecast(inputs=[context_data])
            point_forecast = fallback_forecast[0]
            
        forecast_results[scenario_name] = point_forecast
        print(f"  -> {scenario_name} completed successfully.")

    except Exception as e:
        print(f"  -> ERROR during {scenario_name} inference: {e}")
        # Final safety net: if inference crashes entirely, carry the last known value forward
        forecast_results[scenario_name] = np.full(HORIZON_LEN, target_data[-1])

print("\nPhase 3 Complete: All scenario inferences finished.")

Phase 3 - Part 2: Executing TimesFM Scenarios...
Running inference for: Base Case...
  -> Base Case completed successfully.
Running inference for: Bull Case...
  -> Bull Case completed successfully.
Running inference for: Bear Case...
  -> Bear Case completed successfully.

Phase 3 Complete: All scenario inferences finished.


In [14]:
# ==========================================
# Cell 9: Data Assembly
# ==========================================

print("Phase 4 - Part 1: Assembling Forecast Data...")

# 1. Generate Future Date Index
last_hist_date = df_master['Date'].max()

# Generate the next 12 months, anchoring strictly to End-of-Month (EOM)
future_dates = pd.date_range(
    start=last_hist_date + pd.offsets.MonthBegin(1), 
    periods=HORIZON_LEN, 
    freq='ME'
)
future_dates = future_dates + pd.offsets.MonthEnd(0)

# 2. Create the Forecast DataFrame
df_forecasts = pd.DataFrame({'Date': future_dates})

# Map the TimesFM outputs to their respective columns
for scenario in ['Base Case', 'Bull Case', 'Bear Case']:
    df_forecasts[scenario] = forecast_results[scenario]

# 3. Create a Connection Point for Plotting
# We prepend the final historical data point to the forecasts so the lines connect seamlessly on the chart.
last_val = df_master['PCE_Index'].iloc[-1]
last_historical_row = pd.DataFrame({
    'Date': [last_hist_date],
    'Base Case': [last_val],
    'Bull Case': [last_val],
    'Bear Case': [last_val]
})

df_forecasts_plot = pd.concat([last_historical_row, df_forecasts]).reset_index(drop=True)

print("Forecast data assembled and ready for visualization.")
print(f"Forecast Horizon: {df_forecasts_plot['Date'].min().date()} to {df_forecasts_plot['Date'].max().date()}")

# Display the connected dataframe
display(df_forecasts_plot)

Phase 4 - Part 1: Assembling Forecast Data...
Forecast data assembled and ready for visualization.
Forecast Horizon: 2026-02-28 to 2027-02-28


,Date,Base Case,Bull Case,Bear Case
0,2026-02-28,88.681000,88.681000,88.681000
1,2026-03-31,83.577452,82.280286,86.171784
2,2026-04-30,84.261228,80.953455,87.828436
3,2026-05-31,85.477673,80.289008,88.980022
4,2026-06-30,86.697085,78.914086,89.680568
5,2026-07-31,87.707767,79.665335,90.302100
6,2026-08-31,88.690307,80.907310,90.895490
7,2026-09-30,89.514011,81.731013,91.459760
8,2026-10-31,89.796057,82.142776,91.871523
9,2026-11-30,90.236699,83.102284,92.441882


In [18]:
# ==========================================
# Cell 10: The Dashboard Visualization
# ==========================================

print("Phase 4 - Part 2: Rendering Final Dashboard...")

# Initialize the Plotly figure with 2 subplots (Top: Brent Crude, Bottom: PCE Forecast)
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("Brent Crude Assumptions (USD/bbl)", "DREQRG PCE Index Forecast (Tech & Media Equipment)"),
    row_heights=[0.3, 0.7]
)

last_hist_date = df_master['Date'].max()

# ---------------------------------------------------------
# Top Panel: Brent Crude Scenarios & History
# ---------------------------------------------------------

# Plot Historical Brent Crude Data
fig.add_trace(go.Scatter(
    x=df_master['Date'], 
    y=df_master['Brent_Crude_Price'],
    mode='lines',
    name="Historical Brent Crude",
    line=dict(color='black', width=2),
    legendgroup="Brent"
), row=1, col=1)


# We need to prepend the last historical Brent price to make the forecast lines connect
last_brent_val = df_master['Brent_Crude_Price'].iloc[-1]

for case_name in ['Base Case', 'Bull Case', 'Bear Case']:
    # Add the current month's known price to the front of the scenario array for plotting
    plot_brent_y = np.insert(brent_scenarios[case_name][1:], 0, last_brent_val)
    
    # Determine color based on scenario
    color = 'blue' if case_name == 'Base Case' else ('red' if case_name == 'Bull Case' else 'green')
    
    fig.add_trace(go.Scatter(
        x=df_forecasts_plot['Date'], 
        y=plot_brent_y,
        mode='lines+markers',
        name=f"Brent {case_name}",
        line=dict(color=color, dash='dot'),
        legendgroup="Brent"
    ), row=1, col=1)

# Add a marker for the "Present Day" anchor point
fig.add_trace(go.Scatter(
    x=[last_hist_date], y=[last_brent_val],
    mode='markers', marker=dict(color='black', size=10, symbol='diamond'),
    name="Current Price Anchor", showlegend=False
), row=1, col=1)


# ---------------------------------------------------------
# Bottom Panel: Historical & Forecasted PCE Index
# ---------------------------------------------------------

# Plot Historical DREQRG Data 
fig.add_trace(go.Scatter(
    x=df_master['Date'], 
    y=df_master['PCE_Index'],
    mode='lines',
    name="Historical DREQRG",
    line=dict(color='black', width=2),
    legendgroup="PCE"
), row=2, col=1)

# Plot Forecasts
colors = {'Base Case': 'blue', 'Bull Case': 'red', 'Bear Case': 'green'}
for case_name in ['Base Case', 'Bull Case', 'Bear Case']:
    fig.add_trace(go.Scatter(
        x=df_forecasts_plot['Date'], 
        y=df_forecasts_plot[case_name],
        mode='lines',
        name=f"Forecast ({case_name})",
        line=dict(color=colors[case_name], width=2),
        legendgroup="PCE"
    ), row=2, col=1)


# ---------------------------------------------------------
# Layout & Formatting
# ---------------------------------------------------------

# Add a vertical line to explicitly denote where history ends and forecasting begins
for r in [1, 2]:
    fig.add_vline(x=last_hist_date, line_width=1, line_dash="dash", line_color="gray", row=r, col=1)

# Format the axes and add the required edge-case disclaimer
fig.update_layout(
    height=800,
    title_text="12-Month Macro-Energy Structural Forecast: Tech & Media Equipment PCE",
    title_font_size=20,
    hovermode="x unified",
    template="plotly_white",
    annotations=[
        dict(
            x=0, y=-0.12, xref='paper', yref='paper',
            text="<b>Note:</b> This forecast models tech elasticity purely through the lens of global energy input costs and logistics (Brent Crude).",
            showarrow=False, font=dict(size=11, color="gray"), align="left"
        )
    ]
)

# Update Y-axes labels
fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
fig.update_yaxes(title_text="PCE Index Value", row=2, col=1)

fig.show()

print("\nPipeline Execution Complete.")

Phase 4 - Part 2: Rendering Final Dashboard...



Pipeline Execution Complete.


In [19]:
# ==========================================
# Cell 11: Backtesting Lookback Windows (10 to 60 Years)
# ==========================================

import time

print("Phase 5: Backtesting Context Lengths...")
print("Evaluating model performance across different lookback windows.\n")

# 1. Configuration for Backtest
backtest_windows = [120, 240, 360, 480, 600, 720] # 10 to 60 years in months
BACKTEST_HORIZON = 12

# We need enough historical data to act as "truth" for the backtest
# The target data goes back to 1959. Let's use the most recent 12 months as the "test" set,
# and data prior to that as the "training/context" set.

# Ensure we have enough data overall for the longest lookback
total_required_months = max(backtest_windows) + BACKTEST_HORIZON
if len(df_master) < total_required_months:
     print(f"WARNING: Insufficient historical data ({len(df_master)} months) for maximum backtest window ({total_required_months} months).")
     print("Adjusting backtest windows to available data length.")
     backtest_windows = [w for w in backtest_windows if w + BACKTEST_HORIZON <= len(df_master)]

if not backtest_windows:
     raise ValueError("Not enough data to run any backtests.")

# 2. Define Train/Test Split
# The 'test' set is the final 12 months of the historical dataset
actual_test_data = df_master['PCE_Index'].iloc[-BACKTEST_HORIZON:].values
test_dates = df_master['Date'].iloc[-BACKTEST_HORIZON:]

# The 'training' set is everything before the final 12 months
train_df = df_master.iloc[:-BACKTEST_HORIZON]

results = {}

# 3. Execution Loop
for window in backtest_windows:
    years = window // 12
    print(f"--- Testing {years}-Year Lookback ({window} months) ---")
    start_time = time.time()
    
    # Extract the context data for this specific window size
    context_data = train_df['PCE_Index'].iloc[-window:].values
    
    # We also need the covariate data for the context AND the "future" (which is actually the test period)
    # Context Covariate
    covariate_context = train_df['Brent_Crude_Price'].iloc[-window:].values
    # "Future" Covariate (the actual Brent prices during the 12-month test period)
    covariate_future = df_master['Brent_Crude_Price'].iloc[-BACKTEST_HORIZON:].values
    
    # Combine into full covariate payload
    full_covariate = np.concatenate([covariate_context, covariate_future])
    
    try:
        # We MUST recompile the model for each new context length
        tfm.compile(
            timesfm.ForecastConfig(
                max_context=window,
                max_horizon=BACKTEST_HORIZON,
                normalize_inputs=True,
                infer_is_positive=True,
                return_backcast=True 
            )
        )
        
        # Execute Forecast
        forecast, _ = tfm.forecast_with_covariates(
            inputs=[context_data], 
            dynamic_numerical_covariates={"brent_price": [full_covariate]}
        )
        
        point_forecast = forecast[0]
        
        # Calculate Mean Absolute Error (MAE)
        mae = np.mean(np.abs(point_forecast - actual_test_data))
        results[f"{years} Years"] = {'forecast': point_forecast, 'mae': mae}
        
        elapsed_time = time.time() - start_time
        print(f"    -> MAE: {mae:.4f} (Completed in {elapsed_time:.2f}s)")
        
    except Exception as e:
         print(f"    -> Error during evaluation: {e}")
         results[f"{years} Years"] = {'forecast': np.full(BACKTEST_HORIZON, np.nan), 'mae': np.nan}


# 4. Visualization of Backtest Results
print("\nRendering Backtest Comparison Dashboard...")

fig_backtest = make_subplots(
    rows=1, cols=1,
    subplot_titles=("Backtest: Actual vs. Forecasts by Context Length")
)

# Plot actual test data
fig_backtest.add_trace(go.Scatter(
    x=test_dates, y=actual_test_data,
    mode='lines+markers', name='Actual DREQRG',
    line=dict(color='black', width=3)
))

# Plot each forecast
colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown']
color_idx = 0
for label, data in results.items():
    if not np.isnan(data['mae']): # Only plot successful runs
        fig_backtest.add_trace(go.Scatter(
            x=test_dates, y=data['forecast'],
            mode='lines', name=f"{label} (MAE: {data['mae']:.2f})",
            line=dict(color=colors[color_idx % len(colors)], dash='dash')
        ))
        color_idx += 1

fig_backtest.update_layout(
    height=600,
    title_text="TimesFM Model Tuning: Context Length Evaluation",
    hovermode="x unified",
    template="plotly_white"
)

fig_backtest.update_yaxes(title_text="PCE Index Value")
fig_backtest.show()

print("\nBacktest Complete. Review the MAE scores in the legend to determine the optimal lookback window.")

Phase 5: Backtesting Context Lengths...
Evaluating model performance across different lookback windows.

Adjusting backtest windows to available data length.
--- Testing 10-Year Lookback (120 months) ---
    -> MAE: 1.5119 (Completed in 0.32s)
--- Testing 20-Year Lookback (240 months) ---
    -> MAE: 2.0196 (Completed in 0.67s)
--- Testing 30-Year Lookback (360 months) ---
    -> MAE: 28.5984 (Completed in 0.62s)

Rendering Backtest Comparison Dashboard...



Backtest Complete. Review the MAE scores in the legend to determine the optimal lookback window.
